# Combinatorial mixed volumes and coefficient slices

This notebook uses small examples to illustrate combinatorial mixed volume computations, selected inequalities, and Lorentzian properties.

In [ ]:
from itertools import product
from mixed_ehrhart import (
    lattice_count,
    check_af_inequality,
    check_degree2_inequality,
    check_degree3_inequalities,
    cmv_bivariate,
    combinatorial_mixed_volume,
    degree3_slice_values,
    homogeneous_part,
    is_m_convex_support,
    lorentzian_slice_test,
    multivariate_ehrhart,
    poly_from_vertices,
    polynomial_slice,
    precompute_counts_bivariate,
)

## Inclusion–exclusion for two segments

The convention is

$$\operatorname{CMV}(P,Q)=E(P+Q)-E(P)-E(Q)+1,$$

where $E(S)=\#(S\cap\mathbb Z^d)$. The final $+1$ is the count of the empty
Minkowski sum $\{0\}$. Perpendicular unit segments give $4-2-2+1=1$.
Repeated inputs are kept as separate copies.

In [ ]:
P = poly_from_vertices([(0, 0), (1, 0)])
Q = poly_from_vertices([(0, 0), (0, 1)])
assert combinatorial_mixed_volume([P, Q]) == 1
assert combinatorial_mixed_volume([P, P]) == 0
assert combinatorial_mixed_volume([]) == 1

counts = precompute_counts_bivariate(P, Q, max_a=2, max_b=2)
assert counts[(0, 0)] == 1
assert counts[(2, 1)] == 6
assert cmv_bivariate(counts, 1, 1) == 1
assert cmv_bivariate(counts, 2, 0) == 0
counts

The degree-two diagnostic compares
$\operatorname{CMV}(P,Q)^2$ with
$\operatorname{CMV}(P,P)\operatorname{CMV}(Q,Q)$.
Its return value contains both sides and a boolean; the function does not
print or run a search.

In [ ]:
degree2 = check_degree2_inequality(P, Q)
assert degree2["CMV_PP"] == 0
assert degree2["CMV_PQ"] == 1
assert degree2["CMV_QQ"] == 0
assert degree2["lhs"] == 1
assert degree2["rhs"] == 0
assert degree2["holds"]
degree2

## Repeated unit cubes

For the three-dimensional unit cube $C$, one has
$E(aC+bC)=(a+b+1)^3$. All four order-three forward differences are 6.
For a general pair $P,Q$, their slots are

$$A=\operatorname{CMV}(P,P,P),\quad B=\operatorname{CMV}(P,P,Q),
\quad C=\operatorname{CMV}(P,Q,Q),\quad D=\operatorname{CMV}(Q,Q,Q).$$

The degree-three diagnostics compare $AC\le B^2$ and $BD\le C^2$.

In [ ]:
cube = poly_from_vertices(list(product((0, 1), repeat=3)))
A, B, C, D = degree3_slice_values(cube, cube)
assert (A, B, C, D) == (6, 6, 6, 6)

cube_counts = precompute_counts_bivariate(cube, cube)
assert cmv_bivariate(cube_counts, 2, 1) == 6
assert combinatorial_mixed_volume([cube, cube, cube]) == 6

degree3 = check_degree3_inequalities(cube, cube)
assert degree3["ineq1_lhs"] == degree3["ineq1_rhs"] == 36
assert degree3["ineq2_lhs"] == degree3["ineq2_rhs"] == 36
assert degree3["ineq1_holds"] and degree3["ineq2_holds"]
degree3

The generalized AF-style helper fixes additional polytopes in all three
mixed volumes. Here the extra cube makes all three terms order three.

In [ ]:
af = check_af_inequality(cube, cube, others=[cube])
assert af["mixed"] == af["repeated_P"] == af["repeated_Q"] == 6
assert af["lhs"] == af["rhs"] == 36
assert af["holds"]
af

## Normalized coefficient slices

Let $c_\alpha=\Delta^\alpha E(0)$ be the binomial basis coefficients.
The normalized degree-$m$ slice is

$$f_m(y)=\sum_{|\alpha|=m}\frac{c_\alpha}{\alpha!}y^\alpha.$$


In [ ]:
cube_ehrhart, cube_differences = multivariate_ehrhart([cube, cube], names=["a", "b"])
a, b = cube_ehrhart.parent().gens()
assert cube_ehrhart == (a + b + 1)**3

normalized = polynomial_slice(cube_differences, 3, names=["a", "b"])
raw = polynomial_slice(cube_differences, 3, names=["a", "b"], normalize=False)
assert normalized == (a + b)**3
assert raw == 6 * (a**3 + a**2 * b + a * b**2 + b**3)
assert normalized == homogeneous_part(cube_ehrhart, 3)
normalized, raw

In [ ]:
triangle = poly_from_vertices([(0, 0), (1, 0), (0, 1)])
triangle_ehrhart, triangle_differences = multivariate_ehrhart(
    [triangle, triangle], names=["a", "b"]
)
a, b = triangle_ehrhart.parent().gens()
linear_slice = polynomial_slice(triangle_differences, 1, names=["a", "b"])
assert linear_slice == 2 * (a + b)
assert homogeneous_part(triangle_ehrhart, 1) == 3 * (a + b) / 2
linear_slice

## Exact Lorentzian checks

For a homogeneous polynomial with nonnegative coefficients, check M-convex support and the signatures of quadratic derivatives. For degree $m$, every derivative of total order $m-2$ must have a Hessian with at most one positive eigenvalue. Signatures use exact real algebraic arithmetic.

The support of $(a+b)^3$ has the required exchange property, and the exact
test passes:

In [ ]:
support = {
    alpha for alpha, value in cube_differences.items()
    if sum(alpha) == 3 and value != 0
}
assert is_m_convex_support(support)
holds, failures = lorentzian_slice_test(normalized)
assert holds
assert failures == []
holds, failures

For comparison, $a^2+b^2$ has two positive Hessian eigenvalues and its support
fails exchange:

In [ ]:
a, b = normalized.parent().gens()
holds, failures = lorentzian_slice_test(a**2 + b**2)
assert not holds
assert failures
holds, failures